# 02: Building Eval Datasets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/02_building_eval_datasets.ipynb)

Good metrics cannot rescue a weak dataset. In practice, the dataset is often the highest-leverage part of an eval system because it defines what reality your benchmark is allowed to notice.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Learning goals**
- Understand **Why dataset design is leverage**
- Understand **Start from a candidate task pool**
- Understand **Sample realistic tasks, not just available tasks**
- Understand **Define a label schema before labeling**


## Learning goals

- understand why golden datasets matter more than a few hand-picked examples
- sample realistic tasks instead of only easy or clean tasks
- define label schemas that are clear enough to score
- create train, dev, and test splits for disciplined iteration
- check for contamination and leakage before trusting results
- save a small dataset to JSON and CSV for reuse

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Why dataset design is leverage

A model can only be measured against the data you give it. That means weak datasets produce weak conclusions:

- a narrow dataset rewards brittle prompts
- ambiguous labels create fake disagreement
- leaked examples make systems look stronger than they are
- unrealistic tasks optimize for demos instead of reality

The goal of this notebook is to build a small but thoughtful eval dataset from scratch.

## Start from a candidate task pool

Before you create a golden set, collect a wider pool of realistic tasks. This pool should reflect the channels and situations the system actually sees.

In [ ]:

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        if isinstance(rows[0], dict):
            headers = list(rows[0].keys())
        else:
            headers = [f"col_{index}" for index in range(len(rows[0]))]

    normalized = []
    for row in rows:
        if isinstance(row, dict):
            normalized.append([str(row.get(header, "")) for header in headers])
        else:
            normalized.append([str(value) for value in row])

    widths = []
    for index, header in enumerate(headers):
        content_width = max(len(values[index]) for values in normalized)
        widths.append(max(len(str(header)), content_width))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)

    print(header_line)
    print(divider)
    for values in normalized:
        print(" | ".join(values[index].ljust(widths[index]) for index in range(len(headers))))


In [ ]:

candidate_pool = [
    {"id": "c1", "channel": "email", "product_area": "billing", "text": "We switched to annual billing, but the account still shows monthly invoices."},
    {"id": "c2", "channel": "chat", "product_area": "analytics", "text": "The team dashboard spins forever after I click campaign attribution."},
    {"id": "c3", "channel": "email", "product_area": "workflows", "text": "Can you add approval routing so managers can sign off before purchase orders are sent?"},
    {"id": "c4", "channel": "security_queue", "product_area": "auth", "text": "A former vendor can still access two private workspaces even after offboarding."},
    {"id": "c5", "channel": "chat", "product_area": "auth", "text": "Our password reset emails never arrive for anyone on the finance team."},
    {"id": "c6", "channel": "call_summary", "product_area": "mobile", "text": "Customer reports that large video uploads fail on iOS but not desktop."},
    {"id": "c7", "channel": "email", "product_area": "billing", "text": "Please refund the duplicate renewal charge from this morning."},
    {"id": "c8", "channel": "community_forum", "product_area": "dashboards", "text": "Would love scheduled Slack alerts whenever a KPI drops below target."},
    {"id": "c9", "channel": "security_queue", "product_area": "auth", "text": "We detected repeated login attempts from an unknown country overnight."},
    {"id": "c10", "channel": "chat", "product_area": "integrations", "text": "Single sign-on stopped working after we rotated our identity provider certificate."},
    {"id": "c11", "channel": "email", "product_area": "reports", "text": "Downloaded CSV exports are missing the last two columns from our custom report."},
    {"id": "c12", "channel": "call_summary", "product_area": "workflows", "text": "Prospect wants conditional reminders sent only after legal approval is complete."},
    {"id": "c13", "channel": "email", "product_area": "billing", "text": "Invoice total is wrong because it still includes seats we removed last week."},
    {"id": "c14", "channel": "chat", "product_area": "auth", "text": "A new hire keeps seeing an invalid token message when trying to join via magic link."},
    {"id": "c15", "channel": "community_forum", "product_area": "analytics", "text": "Please add cohort retention charts that group users by acquisition month."},
]

print("Candidate pool size:", len(candidate_pool))
print_table(candidate_pool[:8], headers=["id", "channel", "product_area", "text"])


## Sample realistic tasks, not just available tasks

Convenience sampling is one of the most common eval mistakes. The easiest examples to collect are often the cleanest, shortest, and least ambiguous. Real systems rarely get such friendly traffic.

In [ ]:

from collections import Counter, defaultdict

channel_counts = Counter(item["channel"] for item in candidate_pool)
area_counts = Counter(item["product_area"] for item in candidate_pool)

print("Channel coverage")
print_table([{"channel": key, "count": value} for key, value in sorted(channel_counts.items())])
print()
print("Product area coverage")
print_table([{"product_area": key, "count": value} for key, value in sorted(area_counts.items())])
print()

sampling_plan = [
    {"slice": "billing and billing-adjacent", "target_examples": 3, "why": "Common production workload"},
    {"slice": "auth and security", "target_examples": 4, "why": "Lower volume but high severity"},
    {"slice": "feature requests", "target_examples": 3, "why": "Useful for routing and prioritization"},
    {"slice": "bug reports", "target_examples": 2, "why": "Important for technical support flows"},
]

print("Example sampling plan")
print_table(sampling_plan, headers=["slice", "target_examples", "why"])


## Define a label schema before labeling

A good schema should be:

- specific enough to score
- broad enough to cover reality
- stable across annotators
- connected to a real decision the system must support

We will create a small routing dataset with multiple labels per example.

In [ ]:

label_schema = {
    "intent": {
        "allowed_values": ["billing", "technical_issue", "feature_request", "security", "account_access"],
        "purpose": "Primary routing destination for the ticket"
    },
    "urgency": {
        "allowed_values": ["low", "medium", "high"],
        "purpose": "How quickly a human should review the case"
    },
    "requires_human": {
        "allowed_values": [True, False],
        "purpose": "Whether the issue can be handled automatically"
    },
    "ideal_action": {
        "allowed_values": "free text instruction",
        "purpose": "What the assistant or human team should do next"
    },
}

schema_rows = []
for field_name, details in label_schema.items():
    schema_rows.append(
        {
            "field": field_name,
            "allowed_values": details["allowed_values"],
            "purpose": details["purpose"],
        }
    )

print_table(schema_rows, headers=["field", "allowed_values", "purpose"])


## Build a small golden dataset

The golden dataset is the subset that receives reviewed labels. In a real workflow you would usually store annotation guidelines, reviewer notes, and adjudication decisions alongside the labels.

In [ ]:

gold_records = [
    {
        "id": "g1",
        "source_id": "c1",
        "channel": "email",
        "product_area": "billing",
        "text": "We switched to annual billing, but the account still shows monthly invoices.",
        "intent": "billing",
        "urgency": "medium",
        "requires_human": True,
        "ideal_action": "Verify plan state and correct invoice schedule.",
        "difficulty": "medium"
    },
    {
        "id": "g2",
        "source_id": "c2",
        "channel": "chat",
        "product_area": "analytics",
        "text": "The team dashboard spins forever after I click campaign attribution.",
        "intent": "technical_issue",
        "urgency": "medium",
        "requires_human": True,
        "ideal_action": "Reproduce the dashboard failure and collect logs.",
        "difficulty": "easy"
    },
    {
        "id": "g3",
        "source_id": "c3",
        "channel": "email",
        "product_area": "workflows",
        "text": "Can you add approval routing so managers can sign off before purchase orders are sent?",
        "intent": "feature_request",
        "urgency": "low",
        "requires_human": False,
        "ideal_action": "Route to product feedback queue with workflow context.",
        "difficulty": "easy"
    },
    {
        "id": "g4",
        "source_id": "c4",
        "channel": "security_queue",
        "product_area": "auth",
        "text": "A former vendor can still access two private workspaces even after offboarding.",
        "intent": "security",
        "urgency": "high",
        "requires_human": True,
        "ideal_action": "Escalate to security and revoke access immediately.",
        "difficulty": "medium"
    },
    {
        "id": "g5",
        "source_id": "c5",
        "channel": "chat",
        "product_area": "auth",
        "text": "Our password reset emails never arrive for anyone on the finance team.",
        "intent": "account_access",
        "urgency": "high",
        "requires_human": True,
        "ideal_action": "Check email delivery, spam rules, and reset pipeline health.",
        "difficulty": "medium"
    },
    {
        "id": "g6",
        "source_id": "c6",
        "channel": "call_summary",
        "product_area": "mobile",
        "text": "Customer reports that large video uploads fail on iOS but not desktop.",
        "intent": "technical_issue",
        "urgency": "medium",
        "requires_human": True,
        "ideal_action": "Send to mobile engineering with file size details.",
        "difficulty": "medium"
    },
    {
        "id": "g7",
        "source_id": "c8",
        "channel": "community_forum",
        "product_area": "dashboards",
        "text": "Would love scheduled Slack alerts whenever a KPI drops below target.",
        "intent": "feature_request",
        "urgency": "low",
        "requires_human": False,
        "ideal_action": "Log request in roadmap tracker under alerting.",
        "difficulty": "easy"
    },
    {
        "id": "g8",
        "source_id": "c9",
        "channel": "security_queue",
        "product_area": "auth",
        "text": "We detected repeated login attempts from an unknown country overnight.",
        "intent": "security",
        "urgency": "high",
        "requires_human": True,
        "ideal_action": "Investigate suspicious activity and verify account safety.",
        "difficulty": "hard"
    },
    {
        "id": "g9",
        "source_id": "c10",
        "channel": "chat",
        "product_area": "integrations",
        "text": "Single sign-on stopped working after we rotated our identity provider certificate.",
        "intent": "account_access",
        "urgency": "high",
        "requires_human": True,
        "ideal_action": "Check certificate trust chain and SSO configuration.",
        "difficulty": "hard"
    },
    {
        "id": "g10",
        "source_id": "c11",
        "channel": "email",
        "product_area": "reports",
        "text": "Downloaded CSV exports are missing the last two columns from our custom report.",
        "intent": "technical_issue",
        "urgency": "medium",
        "requires_human": True,
        "ideal_action": "Route to reporting engineering with export template details.",
        "difficulty": "medium"
    },
    {
        "id": "g11",
        "source_id": "c13",
        "channel": "email",
        "product_area": "billing",
        "text": "Invoice total is wrong because it still includes seats we removed last week.",
        "intent": "billing",
        "urgency": "medium",
        "requires_human": True,
        "ideal_action": "Audit seat counts and adjust invoice.",
        "difficulty": "medium"
    },
    {
        "id": "g12",
        "source_id": "c15",
        "channel": "community_forum",
        "product_area": "analytics",
        "text": "Please add cohort retention charts that group users by acquisition month.",
        "intent": "feature_request",
        "urgency": "low",
        "requires_human": False,
        "ideal_action": "Capture request for analytics roadmap review.",
        "difficulty": "medium"
    },
]

print("Golden dataset size:", len(gold_records))
print_table(gold_records[:6], headers=["id", "intent", "urgency", "requires_human", "difficulty", "text"])


## Inspect the labels before you split anything

Early inspection is where you catch label drift, accidental ambiguity, or a missing schema field. Do this before you create train, dev, and test splits.

In [ ]:

intent_counts = Counter(record["intent"] for record in gold_records)
urgency_counts = Counter(record["urgency"] for record in gold_records)

print("Intent distribution")
print_table([{"intent": key, "count": value} for key, value in sorted(intent_counts.items())])
print()
print("Urgency distribution")
print_table([{"urgency": key, "count": value} for key, value in sorted(urgency_counts.items())])


## Create train, dev, and test splits

The split strategy depends on the use case, but the principle is stable:

- train is where you draft ideas
- dev is where you compare and tune
- test is where you check the final claim

Never let the test split quietly become part of your iteration loop.

In [ ]:

def stratified_split(records, label_key, seed=7):
    random.seed(seed)
    grouped = defaultdict(list)
    for record in records:
        grouped[record[label_key]].append(dict(record))

    assigned = []
    for label, items in grouped.items():
        random.shuffle(items)
        total = len(items)

        if total == 1:
            split_plan = ["train"]
        elif total == 2:
            split_plan = ["train", "test"]
        elif total == 3:
            split_plan = ["train", "dev", "test"]
        else:
            dev_count = 1
            test_count = 1
            train_count = total - dev_count - test_count
            split_plan = ["train"] * train_count + ["dev"] * dev_count + ["test"] * test_count

        for item, split in zip(items, split_plan):
            item["split"] = split
            assigned.append(item)

    return sorted(assigned, key=lambda record: record["id"])

assigned_records = stratified_split(gold_records, "intent", seed=11)
print_table(assigned_records, headers=["id", "intent", "split", "difficulty", "text"])


## Contamination and leakage can invalidate the benchmark

Leakage happens when the system has already seen the answer, the example, or a near-duplicate of the example during development or training. Even lightweight contamination checks are better than doing nothing.

In [ ]:

historical_examples = [
    {"id": "h1", "source": "old_regression_suite", "text": "We switched to annual billing but the account still shows monthly invoices."},
    {"id": "h2", "source": "support_playbook", "text": "Single sign-on broke after our identity provider certificate changed."},
    {"id": "h3", "source": "training_notes", "text": "Can you add Slack alerts whenever a KPI drops below target?"},
    {"id": "h4", "source": "old_regression_suite", "text": "Need help resetting my password because reset emails never arrive."},
]

def normalize_text(text):
    lowered = []
    for character in text.lower():
        lowered.append(character if character.isalnum() else " ")
    return " ".join("".join(lowered).split())

def token_set(text):
    return set(normalize_text(text).split())

def jaccard_similarity(left_text, right_text):
    left_tokens = token_set(left_text)
    right_tokens = token_set(right_text)
    if not left_tokens and not right_tokens:
        return 1.0
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)

def leakage_scan(records, prior_examples, threshold=0.65):
    findings = []
    for record in records:
        for previous in prior_examples:
            similarity = jaccard_similarity(record["text"], previous["text"])
            exact_match = normalize_text(record["text"]) == normalize_text(previous["text"])
            if exact_match or similarity >= threshold:
                findings.append(
                    {
                        "record_id": record["id"],
                        "historical_id": previous["id"],
                        "historical_source": previous["source"],
                        "similarity": round(similarity, 3),
                        "exact_match": exact_match,
                    }
                )
    return findings


## Run a simple leakage scan

This scan is intentionally basic, but it already catches exact duplicates and highly overlapping examples. In a larger project you might also check source documents, development prompts, prior benchmark suites, and synthetic data generation inputs.

In [ ]:

leakage_findings = leakage_scan(assigned_records, historical_examples, threshold=0.6)

if leakage_findings:
    print_table(leakage_findings, headers=["record_id", "historical_id", "historical_source", "similarity", "exact_match"])
else:
    print("No potential leakage found.")


## Save the dataset to JSON and CSV

JSON is convenient for nested structures and notebook workflows. CSV is convenient for spreadsheet review and quick audits. Saving both is often worth the extra few lines of code.

In [ ]:

output_dir = Path("module_1_data")
output_dir.mkdir(exist_ok=True)

json_path = output_dir / "02_eval_dataset.json"
csv_path = output_dir / "02_eval_dataset.csv"

with json_path.open("w", encoding="utf-8") as handle:
    json.dump(assigned_records, handle, indent=2)

fieldnames = list(assigned_records[0].keys())
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(assigned_records)

print("Saved:", json_path)
print("Saved:", csv_path)


## Reload and inspect the saved artifacts

Round-tripping the dataset is a simple but useful check. It confirms that your saved artifact matches what you think you created.

In [ ]:

with (Path("module_1_data") / "02_eval_dataset.json").open("r", encoding="utf-8") as handle:
    reloaded_json = json.load(handle)

with (Path("module_1_data") / "02_eval_dataset.csv").open("r", encoding="utf-8") as handle:
    reloaded_csv = list(csv.DictReader(handle))

print("Reloaded JSON rows:", len(reloaded_json))
print("Reloaded CSV rows:", len(reloaded_csv))
print()
print_table(reloaded_json[:5], headers=["id", "intent", "split", "urgency", "text"])


## Build a small dataset dashboard

Even a simple dataset deserves a quick dashboard. This is how you verify that labels, splits, and example difficulty are roughly aligned with your plan.

In [ ]:

split_counts = Counter(record["split"] for record in assigned_records)
difficulty_by_split = defaultdict(Counter)

for record in assigned_records:
    difficulty_by_split[record["split"]][record["difficulty"]] += 1

print("Split counts")
print_table([{"split": key, "count": value} for key, value in sorted(split_counts.items())])
print()

difficulty_rows = []
for split, counter in sorted(difficulty_by_split.items()):
    row = {"split": split}
    for difficulty in ["easy", "medium", "hard"]:
        row[difficulty] = counter.get(difficulty, 0)
    difficulty_rows.append(row)

print("Difficulty by split")
print_table(difficulty_rows, headers=["split", "easy", "medium", "hard"])


## Dataset-building checklist

- collect from realistic channels
- define the schema before large-scale labeling
- inspect examples before trusting the labels
- create disciplined splits
- scan for leakage
- save artifacts in stable formats

In the next notebook we will turn datasets into numbers by implementing metrics from scratch.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📝 Key Takeaways

- **Learning goals** — revisit this section for reference
- **Why dataset design is leverage** — revisit this section for reference
- **Start from a candidate task pool** — revisit this section for reference
- **Sample realistic tasks, not just available tasks** — revisit this section for reference
- **Define a label schema before labeling** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory